# W02 – SQL esencial I en DuckDB (SELECT/WHERE/GROUP BY/NULLs)

## Conexión con DDIA
- **DDIA Cap. 2**: modelos de datos y lenguajes de consulta (SQL como herramienta central).
- Aquí convertimos *preguntas* en *consultas* sobre un dataset real (NASA Exoplanet Archive).

## Prerrequisitos
- Haber hecho W01A y W01B (o al menos tener Python + dependencias instaladas).
- Si no tienes `data/raw/pscomppars.csv`, el notebook lo descargará.

## Objetivos
- Crear una **vista** `raw_ps` desde un CSV.
- Usar SQL básico: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `GROUP BY`, `HAVING`.
- Entender `NULL`: `COUNT(*)` vs `COUNT(col)` y `COALESCE`.

## Checklist de evidencias
- [ ] Output de: `SELECT count(*) FROM raw_ps`
- [ ] 6 consultas resueltas en la sección **TU TURNO**
- [ ] 10 consultas adicionales (tarea) guardadas al final


In [1]:
import os
os.chdir("..")
os.getcwd()

'/home/manuela-rios/Documentos/Física computacional/Manuela-Rios'

In [2]:
# Setup común (cross-platform)
import sys, subprocess
from pathlib import Path
import duckdb

DB_PATH = Path("data/exoplanets.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH))

def run_module(mod: str, *args: str):
    cmd = [sys.executable, "-m", mod, *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

raw_csv = Path("data/raw/pscomppars.csv")
if not raw_csv.exists():
    # Para clase: descarga razonable. Quita --limit si quieres el subset completo.
    run_module("src.ingest.download_exoplanets", "--format", "csv", "--limit", "50000")

# DuckDB no permite parámetros preparados en DDL (ej. CREATE VIEW).
# Insertamos la ruta como literal SQL, escapando comillas simples.
def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

raw_csv_abs = raw_csv.resolve()
con.execute(
    f"CREATE OR REPLACE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_quote(raw_csv_abs.as_posix())})"
)
con.execute("SELECT count(*) AS n_rows FROM raw_ps").fetchall()


[(6087,)]

## DEMO

In [3]:
# DEMO 1: inspección rápida
con.sql("DESCRIBE raw_ps").show()


┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

In [4]:
# DEMO 2: SELECT + LIMIT (muestra pequeña)
con.sql("""
SELECT pl_name, hostname, discoverymethod, disc_year
FROM raw_ps
LIMIT 10
""").show()


┌───────────────┬─────────────┬─────────────────┬───────────┐
│    pl_name    │  hostname   │ discoverymethod │ disc_year │
│    varchar    │   varchar   │     varchar     │   int64   │
├───────────────┼─────────────┼─────────────────┼───────────┤
│ Kepler-1167 b │ Kepler-1167 │ Transit         │      2016 │
│ Kepler-1740 b │ Kepler-1740 │ Transit         │      2021 │
│ Kepler-1581 b │ Kepler-1581 │ Transit         │      2016 │
│ Kepler-644 b  │ Kepler-644  │ Transit         │      2016 │
│ Kepler-1752 b │ Kepler-1752 │ Transit         │      2021 │
│ Kepler-280 c  │ Kepler-280  │ Transit         │      2014 │
│ Kepler-1208 b │ Kepler-1208 │ Transit         │      2016 │
│ Kepler-263 c  │ Kepler-263  │ Transit         │      2014 │
│ Kepler-1101 b │ Kepler-1101 │ Transit         │      2016 │
│ HD 168746 b   │ HD 168746   │ Radial Velocity │      2002 │
└───────────────┴─────────────┴─────────────────┴───────────┘
  10 rows                                         4 columns



In [5]:
# DEMO 3: WHERE + ORDER BY (evita NULL)
con.sql("""
SELECT pl_name, pl_orbper, pl_rade
FROM raw_ps
WHERE pl_orbper IS NOT NULL
ORDER BY pl_orbper ASC
LIMIT 10
""").show()


┌──────────────────┬─────────────┬─────────────┐
│     pl_name      │  pl_orbper  │   pl_rade   │
│     varchar      │   double    │   double    │
├──────────────────┼─────────────┼─────────────┤
│ PSR J1719-1438 b │ 0.090706293 │        NULL │
│ ZTF J1828+2308 b │   0.1120067 │ 11.13051784 │
│ M62H b           │ 0.132935028 │        NULL │
│ KOI-1843.03      │   0.1768913 │        0.61 │
│ K2-137 b         │    0.179719 │        0.64 │
│ KIC 10001893 b   │      0.2197 │        NULL │
│ ZTF J1230-2655 b │  0.23597766 │ 13.78704626 │
│ TOI-6255 b       │  0.23818244 │       1.079 │
│ KOI-55 b         │    0.240104 │       0.759 │
│ TOI-6324 b       │    0.279221 │       1.059 │
└──────────────────┴─────────────┴─────────────┘
  10 rows                            3 columns



In [6]:
# DEMO 4: NULLs — COUNT(*) vs COUNT(col)
con.sql("""
SELECT
  COUNT(*)                    AS total_rows,
  COUNT(pl_rade)              AS non_null_radius,
  COUNT(*) - COUNT(pl_rade)   AS null_radius
FROM raw_ps
""").show()


┌────────────┬─────────────────┬─────────────┐
│ total_rows │ non_null_radius │ null_radius │
│   int64    │      int64      │    int64    │
├────────────┼─────────────────┼─────────────┤
│       6087 │            6037 │          50 │
└────────────┴─────────────────┴─────────────┘



In [7]:
# DEMO 5: GROUP BY + agregados
con.sql("""
SELECT
  discoverymethod,
  COUNT(*) AS n_planets,
  AVG(pl_rade) AS avg_radius_earth
FROM raw_ps
GROUP BY 1
ORDER BY n_planets DESC
LIMIT 10
""").show()


┌───────────────────────────────┬───────────┬────────────────────┐
│        discoverymethod        │ n_planets │  avg_radius_earth  │
│            varchar            │   int64   │       double       │
├───────────────────────────────┼───────────┼────────────────────┤
│ Transit                       │      4488 │ 4.3714102161533175 │
│ Radial Velocity               │      1161 │   9.79150910617438 │
│ Microlensing                  │       265 │  9.860037735849057 │
│ Imaging                       │        91 │ 15.649137814367812 │
│ Transit Timing Variations     │        39 │  6.493408364210527 │
│ Eclipse Timing Variations     │        17 │ 12.893333333333334 │
│ Orbital Brightness Modulation │         9 │            9.64504 │
│ Pulsar Timing                 │         8 │  5.411333333333332 │
│ Astrometry                    │         6 │ 12.450000000000001 │
│ Pulsation Timing Variations   │         2 │              12.75 │
└───────────────────────────────┴───────────┴─────────────────

In [8]:
# DEMO 6: HAVING (filtra grupos después de agrupar)
con.sql("""
SELECT
  disc_year,
  COUNT(*) AS n
FROM raw_ps
WHERE disc_year IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 200
ORDER BY disc_year ASC
""").show()


┌───────────┬───────┐
│ disc_year │   n   │
│   int64   │ int64 │
├───────────┼───────┤
│      2014 │   869 │
│      2016 │  1496 │
│      2018 │   315 │
│      2020 │   235 │
│      2021 │   554 │
│      2022 │   369 │
│      2023 │   326 │
│      2024 │   260 │
│      2025 │   241 │
└───────────┴───────┘



### Resumen
- `GROUP BY` cambia el grano: ya no son planetas, son **grupos**.
- `COUNT(*)` cuenta filas; `COUNT(col)` ignora `NULL`.
- `HAVING` filtra **después** de agrupar (a diferencia de `WHERE`).

---

## TU TURNO (práctica guiada)
Resuelve estas consultas. Pega el output (al menos las primeras filas) en cada celda.


### 1) ¿Cuántos planetas hay por año? (top 15 años con más planetas)

In [9]:
# TODO (1): ¿Cuántos planetas hay por año? (top 15 años con más planetas)
# Pistas: usa disc_year, filtra IS NOT NULL, GROUP BY, ORDER BY n DESC, LIMIT 15
query = """
SELECT
  disc_year,
  COUNT(*) AS n
FROM raw_ps
WHERE disc_year IS NOT NULL
GROUP BY disc_year
ORDER BY n DESC
LIMIT 15
"""
con.execute(query).fetchall()


[(2016, 1496),
 (2014, 869),
 (2021, 554),
 (2022, 369),
 (2023, 326),
 (2018, 315),
 (2024, 260),
 (2025, 241),
 (2020, 235),
 (2019, 196),
 (2015, 155),
 (2017, 152),
 (2012, 139),
 (2011, 135),
 (2013, 128)]

### 2) Top 10 sistemas (hostname) con más planetas

In [10]:
# TODO (2): Top 10 sistemas (hostname) con más planetas
# Pistas: GROUP BY hostname, cuenta filas, ORDER BY DESC, LIMIT 10
query = """
SELECT
  hostname,
  COUNT(*) AS n_planets
FROM raw_ps
GROUP BY hostname
ORDER BY n_planets DESC
LIMIT 10
"""
con.execute(query).fetchall()


[('KOI-351', 8),
 ('TRAPPIST-1', 7),
 ('HD 34445', 6),
 ('HIP 41378', 6),
 ('HD 191939', 6),
 ('Kepler-11', 6),
 ('TOI-178', 6),
 ('HD 219134', 6),
 ('Kepler-20', 6),
 ('K2-138', 6)]

### 3) ¿Qué fracción de filas tiene `pl_bmasse` nulo?

In [11]:
# TODO (3): ¿Qué fracción de filas tiene pl_bmasse nulo?
# Pistas: COUNT(*) total, COUNT(pl_bmasse) non_null, nulls = total - non_null
#       fracción = nulls / total (convierte a DOUBLE)
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(pl_bmasse) AS non_null_mass,
  (COUNT(*) - COUNT(pl_bmasse)) AS null_mass,
  CAST(COUNT(*) - COUNT(pl_bmasse) AS DOUBLE) / COUNT(*) AS fraction_null
FROM raw_ps
"""
con.execute(query).fetchall()


[(6087, 6056, 31, 0.00509282076556596)]

### 4) 10 planetas con mayor radio (pl_rade) (evita NULL)

In [12]:
# TODO (4): 10 planetas con mayor radio (pl_rade) (evita NULL)
# Pistas: WHERE pl_rade IS NOT NULL, ORDER BY pl_rade DESC, LIMIT 10
query = """
SELECT
  pl_name,
  pl_rade
FROM raw_ps
WHERE pl_rade IS NOT NULL
ORDER BY pl_rade DESC
LIMIT 10
"""
con.execute(query).fetchall()


[('V2376 Ori b', 87.20586985),
 ('HD 100546 b', 77.3421),
 ('GQ Lup b', 33.6),
 ('Kepler-297 d', 32.6),
 ('PDS 70 b', 30.48848),
 ('DH Tau b', 30.2643),
 ('Kepler-1979 b', 29.33),
 ('TOI-1408 b', 25.0),
 ('CT Cha b', 24.66),
 ('TOI-3540 A b', 23.53885947)]

### 5) Compara `COUNT(*)` vs `COUNT(disc_year)` por método

In [13]:
# TODO (5): Compara COUNT(*) vs COUNT(disc_year) por método
# Pistas: GROUP BY discoverymethod, calcula total y non_null_year = COUNT(disc_year)
query = """
SELECT
  discoverymethod,
  COUNT(*) AS total_rows,
  COUNT(disc_year) AS non_null_year
FROM raw_ps
GROUP BY discoverymethod
ORDER BY total_rows DESC
"""
con.execute(query).fetchall()


[('Transit', 4488, 4487),
 ('Radial Velocity', 1161, 1161),
 ('Microlensing', 265, 265),
 ('Imaging', 91, 91),
 ('Transit Timing Variations', 39, 39),
 ('Eclipse Timing Variations', 17, 17),
 ('Orbital Brightness Modulation', 9, 9),
 ('Pulsar Timing', 8, 8),
 ('Astrometry', 6, 6),
 ('Pulsation Timing Variations', 2, 2),
 ('Disk Kinematics', 1, 1)]

### 6) Resumen: por método, n_planets y mediana de periodo orbital

In [14]:
# TODO (6): Resumen por método: n_planets y mediana del periodo orbital
# Pistas: MEDIAN(pl_orbper) (filtra NULL si aplica), GROUP BY discoverymethod
query = """
SELECT
  discoverymethod,
  COUNT(*) AS n_planets,
  MEDIAN(pl_orbper) AS median_orbital_period
FROM raw_ps
WHERE pl_orbper IS NOT NULL
GROUP BY discoverymethod
ORDER BY n_planets DESC
"""
con.execute(query).fetchall()


[('Transit', 4488, 8.15909335),
 ('Radial Velocity', 1161, 305.5),
 ('Transit Timing Variations', 39, 30.0),
 ('Imaging', 25, 33000.0),
 ('Eclipse Timing Variations', 17, 3160.0),
 ('Microlensing', 12, 3142.5),
 ('Orbital Brightness Modulation', 9, 0.81161),
 ('Pulsar Timing', 7, 25.262),
 ('Astrometry', 6, 334.76),
 ('Pulsation Timing Variations', 2, 1005.0)]

## Para entregar (tarea)
1) **4 consultas adicionales** (tú decides las preguntas), pero deben incluir:
   - 2 consultas de calidad (nulos, rangos, duplicados, outliers simples)
   - 2 consultas científicas: pregunta + 1–2 líneas de interpretación 

2) En `docs/decisions_log.md`: 1 decisión de hoy (con evidencia: conteos o query).

## Reflexión (bitácora)
- ¿Qué consulta te pareció más difícil y por qué?
- Si el dataset creciera 100×, ¿qué consultas crees que empeoran más?


**Entrega sugerida:** crea `docs/w02a_sql_practice.md` y pega tus 6 respuestas (1–6) + 4 consultas adicionales tuyas con resultados.


## SOLUCIÓN TAREA

En esta versión rehice la sección final con preguntas nuevas y un enfoque distinto. Separé dos consultas de calidad y dos consultas analíticas, cada una con una interpretación breve.


### Consulta de calidad 1: registros sin nombre de planeta o sin hostname

Esta consulta sirve para detectar filas que podrían dificultar trazabilidad o futuras uniones si faltan identificadores básicos.


In [15]:
query = '''
SELECT
  SUM(CASE WHEN pl_name IS NULL THEN 1 ELSE 0 END) AS null_pl_name,
  SUM(CASE WHEN hostname IS NULL THEN 1 ELSE 0 END) AS null_hostname
FROM raw_ps
'''
con.execute(query).fetchdf()


,null_pl_name,null_hostname
0,0.0,0.0


### Consulta de calidad 2: métodos de descubrimiento poco frecuentes

Busco categorías con muy pocos registros porque suelen ser candidatas a revisión manual o a problemas de estandarización en el texto.


In [16]:
query = '''
SELECT
  discoverymethod,
  COUNT(*) AS n_rows
FROM raw_ps
WHERE discoverymethod IS NOT NULL
GROUP BY discoverymethod
HAVING COUNT(*) <= 3
ORDER BY n_rows ASC, discoverymethod ASC
'''
con.execute(query).fetchdf()


,discoverymethod,n_rows
0,Disk Kinematics,1
1,Pulsation Timing Variations,2


### Consulta analítica 1: ¿qué métodos concentran planetas con menor periodo orbital mediano?

La idea es comparar qué métodos aparecen más asociados a planetas con órbitas cortas.


In [17]:
query = '''
SELECT
  discoverymethod,
  COUNT(*) AS n_planets,
  MEDIAN(pl_orbper) AS median_orbper
FROM raw_ps
WHERE pl_orbper IS NOT NULL
GROUP BY discoverymethod
HAVING COUNT(*) >= 5
ORDER BY median_orbper ASC
LIMIT 10
'''
con.execute(query).fetchdf()


,discoverymethod,n_planets,median_orbper
0,Orbital Brightness Modulation,9,0.811610
1,Transit,4488,8.159093
2,Pulsar Timing,7,25.262000
3,Transit Timing Variations,39,30.000000
4,Radial Velocity,1161,305.500000
5,Astrometry,6,334.760000
6,Microlensing,12,3142.500000
7,Eclipse Timing Variations,17,3160.000000
8,Imaging,25,33000.000000


**Interpretación:** esta consulta ayuda a ver si ciertos métodos tienden a detectar con mayor facilidad planetas de órbita corta, lo cual también puede reflejar sesgos observacionales del instrumento o de la técnica.


### Consulta analítica 2: años con más descubrimientos y masa promedio reportada

Aquí comparo productividad anual con una propiedad física disponible (`pl_bmasse`) para ver si los años más activos también tienen mejor cobertura de mediciones.


In [18]:
query = '''
SELECT
  disc_year,
  COUNT(*) AS n_discoveries,
  AVG(pl_bmasse) AS avg_mass_earth
FROM raw_ps
WHERE disc_year IS NOT NULL
GROUP BY disc_year
HAVING COUNT(*) >= 20
ORDER BY n_discoveries DESC, disc_year DESC
LIMIT 10
'''
con.execute(query).fetchdf()


,disc_year,n_discoveries,avg_mass_earth
0,2016,1496,82.815316
1,2014,869,113.160300
2,2021,554,250.926985
3,2022,369,843.463596
4,2023,326,546.722239
5,2018,315,364.773152
6,2024,260,597.532464
7,2025,241,618.960981
8,2020,235,255.101218
9,2019,196,464.409946


**Interpretación:** los años con más descubrimientos pueden mostrar cambios en capacidad observacional, campañas más intensivas o mejoras en procesamiento de datos; la masa promedio disponible también sugiere qué tan completas estaban las mediciones en esos periodos.


### Decisión para `docs/decisions_log.md`

```markdown
- Fecha: 2026-04-04
- Decisión: Filtrar consultas comparativas por columnas no nulas antes de calcular métricas agregadas.
- Razón: Comparar métodos o años con datos faltantes puede producir conclusiones engañosas si no se controla qué filas realmente aportan a la métrica.
- Alternativas: agregar todo sin filtrar (rechazada), imputar valores faltantes en esta etapa exploratoria (rechazada).
- Evidencia: conteos por `discoverymethod`, `COUNT(*)` vs columnas no nulas, y resultados de consultas agregadas sobre `pl_orbper` y `pl_bmasse`.
```
